# NeoNatal Watch AI — Phase 4: Feature Engineering

> **DEMO / SYNTHETIC DATA — NOT FOR CLINICAL USE**
>
> This notebook demonstrates feature engineering on the preprocessed dataset. 
> These features are essential for traditional models (like XGBoost) that don't natively understand time-series sequences.

---

## What We're Building:
1. **Rolling Statistics**: Mean, Standard Deviation, Min, Max over 15, 30, and 60-minute windows.
2. **Rate of Change**: The difference between the current value and the value X minutes ago.
3. **Patient Isolation**: Ensuring calculations don't bleed across different patient timelines.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ml.features.features import engineer_features

print('Imports OK')

In [ ]:
# Load one of the preprocessed splits (e.g., train)
df_train = pd.read_csv('../data/processed/train.csv', parse_dates=['timestamp'])
print(f"Original columns: {len(df_train.columns)}")

# Apply feature engineering
df_features = engineer_features(df_train, window_sizes=[15, 30, 60])
print(f"New columns: {len(df_features.columns)}")

In [ ]:
# Let's visualize how a rolling feature looks compared to the raw vital
pid = df_features['patient_id'].iloc[0]  # Pick the first patient
sample = df_features[df_features['patient_id'] == pid].head(200)

plt.figure(figsize=(14, 6))
plt.plot(sample['timestamp'], sample['heart_rate'], label='Raw Heart Rate', alpha=0.5)
plt.plot(sample['timestamp'], sample['heart_rate_mean_15m'], label='15m Rolling Mean', linewidth=2)
plt.plot(sample['timestamp'], sample['heart_rate_mean_60m'], label='60m Rolling Mean', linewidth=2)
plt.title(f"Heart Rate vs Rolling Means (Patient {pid})")
plt.xlabel("Time")
plt.ylabel("Heart Rate (bpm)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Visualizing Rate of Change (Slope)
plt.figure(figsize=(14, 4))
plt.plot(sample['timestamp'], sample['heart_rate_diff_15m'], color='purple', label='15m Rate of Change')
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title(f"Heart Rate 15m Rate of Change (Patient {pid})")
plt.ylabel("Difference (bpm)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Check feature correlations with the target label
target = 'deterioration_label'
hr_features = [c for c in df_features.columns if 'heart_rate' in c] + [target]

corr = df_features[hr_features].corr()[target].sort_values(ascending=False)
print("Correlation of Heart Rate Features with Deterioration Label:")
print(corr)